In [1]:
# --- Implementação em Python Puro (Sem bibliotecas externas) ---

def thomas_algorithm(e, f, g, r):
    """
    Algoritmo de Thomas para resolver sistema tridiagonal [cite: 199-201].
    e: subdiagonal, f: diagonal, g: superdiagonal, r: termo de fonte.
    """
    n = len(r)
    # Cópia das listas para não modificar os originais
    e_ = list(e)
    f_ = list(f)
    g_ = list(g)
    r_ = list(r)
    x = [0.0] * n
    
    # Forward elimination (Decomposição L.U) [cite: 204-208]
    for k in range(1, n):
        m = e_[k] / f_[k-1]
        f_[k] -= m * g_[k-1]
        r_[k] -= m * r_[k-1]
        
    # Backward substitution (Substituição Regressiva) [cite: 213-218]
    x[n-1] = r_[n-1] / f_[n-1]
    for k in range(n-2, -1, -1):
        x[k] = (r_[k] - g_[k] * x[k+1]) / f_[k]
    return x

# --- Parâmetros Físicos e Discretização ---
L = 0.01          # Meia espessura [m]
k = 3.0           # Condutividade [W/m.K]
rho = 10500.0     # Densidade [kg/m³]
Cp = 300.0        # Calor específico [J/kg.K]
alpha = k / (rho * Cp)
h = 10000.0       # Coeficiente convectivo
T_inf = 300.0     # Temperatura fluido [°C]
q_dot = 1e8       # Geração interna [W/m³]

N = 21            # Nós
dx = L / (N - 1)
dt = 0.5          # Passo de tempo [s]
t_final = 20.0
Fo = (alpha * dt) / (dx**2)
Bi = (h * dx) / k

# --- Montagem da Matriz Tridiagonal (listas nativas) ---
e = [-Fo] * N
f = [1 + 2*Fo] * N
g = [-Fo] * N

# Contorno (Nó 0 = Simetria, Nó N-1 = Convecção) [cite: 438-440, 420-436]
f[0] = 1 + 2*Fo
g[0] = -2*Fo
e[N-1] = -2*Fo
f[N-1] = 1 + 2*Fo + 2*Fo*Bi

T = [T_inf] * N
A_term = (q_dot / (rho * Cp)) * dt

# --- Loop Temporal ---
for _ in range(int(t_final / dt)):
    b = [T[i] + A_term for i in range(N)]
    b[N-1] = T[N-1] + A_term + (2 * Fo * Bi * T_inf)
    T = thomas_algorithm(e, f, g, b)

# --- Salvar resultados para visualização externa ---
with open("resultados_temperatura.txt", "w") as f_out:
    f_out.write("Distancia(cm),Temperatura(C)\n")
    for i in range(N):
        dist = (i * dx) * 100
        f_out.write(f"{dist:.2f},{T[i]:.2f}\n")

print("Simulação concluída. Resultados salvos em 'resultados_temperatura.txt'.")

Simulação concluída. Resultados salvos em 'resultados_temperatura.txt'.
